***5) CLINICAL SEVERITY ANALYSIS***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "03A_GSE135779_CLINICAL_METADATA")


***5A) CLINICAL METADATA INTEGRATION***

In [ ]:
import pandas as pd

# ---- GSM ID guide: GSM -> patient name [JB code] ----
raw = pd.read_excel(f"{BASE_DIR}/GSE135779 GSM IDs.xlsx", sheet_name="Sheet1", header=None)

pairs = []
for col_start in range(0, raw.shape[1], 3):
    gsm_col = raw.iloc[:, col_start]
    if col_start + 1 >= raw.shape[1]:
        continue
    name_col = raw.iloc[:, col_start + 1]
    for gsm, name in zip(gsm_col, name_col):
        if pd.notna(gsm) and pd.notna(name):
            pairs.append((str(gsm).strip(), str(name).strip()))

guide = pd.DataFrame(pairs, columns=["gsm_id", "name_code"])

# split "cSLE1 [JB17001]" into patient_name="cSLE1", jb_code="JB17001"
guide["patient_name"] = guide["name_code"].str.extract(r"^(\S+)\s*\[")
guide["jb_code"] = guide["name_code"].str.extract(r"\[(\S+)\]")
guide["sample"] = guide["gsm_id"] + "_" + guide["jb_code"]

print("Total GSM-patient pairs:", len(guide))
guide.head(10)

In [ ]:
# ---- Clinical table: patient name -> SLEDAI + other clinical fields ----
clin_raw = pd.read_excel(f"{BASE_DIR}/suppdata.xlsx", sheet_name="ST1b-Clinical information ", header=None)

# locate the header row (contains "Names")
header_row = clin_raw[clin_raw.iloc[:, 0] == "Names"].index[0]
clin = pd.read_excel(f"{BASE_DIR}/suppdata.xlsx", sheet_name="ST1b-Clinical information ", header=header_row)

keep_cols = ["Names", "Groups", "Batch", "Age", "Gender", "SLEDAI", "dsDNA", "low_com", "C3", "C4", "Race", "Ethnicity", "MMF", "OS", "MTX", "Plaquenil", "Collection_year"]
keep_cols = [c for c in keep_cols if c in clin.columns]
clin = clin[keep_cols].rename(columns={"Names": "patient_name"})

print("Clinical table shape:", clin.shape)
clin.head(10)

**Known cohort-size gap.** The GSM ID guide spreadsheet (`GSE135779 GSM IDs.xlsx`) only maps 12 adult patients (7 aSLE + 5 aHD), while the clinical table (`suppdata.xlsx`) lists 14 (8 aSLE + 6 aHD) -- two patients (`aHD2`, `aSLE8`) simply have no GSM mapping in the source spreadsheet available for this project. This is a gap in the source data files themselves, not a code bug, but it means every downstream adult-cohort analysis in this notebook and in 03B and 06A-06B genuinely covers **5 healthy / 7 SLE adults, not 6/8** -- this should be stated explicitly wherever adult sample sizes are reported in the paper, and is an important caveat on statistical power for anything involving the adult arm (see 06A/06B, which already print this power caveat for the age x severity interaction test).

In [ ]:
from pathlib import Path
# ---- Merge: GSM/sample <-> patient name <-> clinical info ----
metadata = guide.merge(clin, on="patient_name", how="left")

# SLEDAI: "ND" (not done / not applicable, e.g. healthy controls) -> NaN, else numeric
metadata["SLEDAI"] = pd.to_numeric(metadata["SLEDAI"], errors="coerce")

out_path = f"{BASE_DIR}/Results/severity_analysis/patient_clinical_metadata.csv"
Path(out_path).parent.mkdir(parents=True, exist_ok=True)
metadata.to_csv(out_path, index=False)

print("Saved:", out_path)
print("\nTotal patients:", len(metadata))
print("\nBy group:")
print(metadata["Groups"].value_counts())
print("\nSLEDAI availability by group:")
print(metadata.groupby("Groups")["SLEDAI"].apply(lambda s: s.notna().sum()))
metadata.head(10)

In [ ]:
from pathlib import Path

audit_dir = Path(BASE_DIR) / "Results" / "qc"
audit_dir.mkdir(parents=True, exist_ok=True)

clinical_fields = [c for c in ["Age", "Gender", "Batch", "SLEDAI", "dsDNA", "low_com", "C3", "C4"] if c in metadata.columns]
missingness = (
    metadata.groupby("Groups", observed=True)[clinical_fields]
    .agg(lambda values: int(values.isna().sum()))
    .reset_index()
)
missingness.to_csv(audit_dir / "clinical_missingness_by_group.csv", index=False)

categorical_rows = []
for field in [c for c in ["Gender", "Batch"] if c in metadata.columns]:
    counts = metadata.groupby(["Groups", field], observed=True).size().rename("n").reset_index()
    counts.insert(1, "field", field)
    counts = counts.rename(columns={field: "level"})
    categorical_rows.append(counts)
confounder_table = pd.concat(categorical_rows, ignore_index=True) if categorical_rows else pd.DataFrame()
confounder_table.to_csv(audit_dir / "potential_confounders_by_group.csv", index=False)

print("Saved clinical missingness and potential-confounder summaries.")
display(missingness)
display(confounder_table)


In [ ]:
# Explicit disclosure of the GSM-mapping gap vs. the full clinical cohort
adult_clin_names = set(clin.loc[clin["patient_name"].astype(str).str.startswith(("aHD", "aSLE")), "patient_name"])
adult_guide_names = set(guide.loc[guide["patient_name"].astype(str).str.startswith(("aHD", "aSLE")), "patient_name"])
missing_from_guide = sorted(adult_clin_names - adult_guide_names)

print(f"Adult patients in clinical table: {len(adult_clin_names)}")
print(f"Adult patients with a GSM mapping (usable in this analysis): {len(adult_guide_names)}")
print(f"Patients present in the clinical table but with NO GSM mapping (excluded from all downstream analysis): {missing_from_guide}")

**SLEDAI distribution by group.**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_df = metadata.dropna(subset=["SLEDAI"])
for group, color in [("cSLE", "#5B3A73"), ("aSLE", "#2C6E68")]:
    vals = plot_df.loc[plot_df["Groups"] == group, "SLEDAI"]
    ax.hist(vals, bins=range(0, 22, 2), alpha=0.6, label=f"{group} (n={len(vals)})", color=color)
ax.set_xlabel("SLEDAI")
ax.set_ylabel("Number of patients")
ax.set_title("SLEDAI distribution: cSLE vs aSLE")
ax.legend()
plt.tight_layout()
out_path = f"{BASE_DIR}/Results/severity_analysis/sledai_distribution.png"
plt.savefig(out_path, dpi=600)
plt.show()
print("Saved:", out_path)